In [1]:
import pandas as pd
import numpy as np
from itertools import permutations
from statsmodels.tsa.stattools import coint

In [2]:
filepath = "/Users/benjaminsumacbookm4/Desktop/Pairs Trading/data/market_data/IntradayMarketDataNVDA_AMD"
data = pd.read_parquet(filepath)

In [3]:
data.head()

,symbol,timestamp,close
0,AMD,2020-11-23 09:00:00+00:00,4.442651
1,AMD,2020-11-23 09:12:00+00:00,4.443827
2,AMD,2020-11-23 09:24:00+00:00,4.443827
3,AMD,2020-11-23 09:42:00+00:00,4.444767
4,AMD,2020-11-23 09:46:00+00:00,4.444767


In [4]:
data.tail()

,symbol,timestamp,close
1485917,NVDA,2024-11-20 22:56:00+00:00,4.969258
1485918,NVDA,2024-11-20 22:57:00+00:00,4.970648
1485919,NVDA,2024-11-20 22:58:00+00:00,4.970924
1485920,NVDA,2024-11-20 22:59:00+00:00,4.970508
1485921,NVDA,2024-11-21 00:00:00+00:00,4.966551


In [6]:
# Show timestamps in New York time so the session is 09:30–16:00 year-round (handles DST)
data["timestamp"] = data["timestamp"].dt.tz_convert("America/New_York")

prices = (
    data.pivot(index="timestamp", columns="symbol", values="close")
        .between_time("09:30", "15:59") # Normalizes time to regular business hours
        .dropna()
)
prices.head()
prices["NVDA"].shape

(391130,)

- score = more negative means more likely to be not cointegrated
- pvalue = Smaller value rejects null hypothesis of lack of cointegration
- crit_value = depends if 0, 1, or 2. Rejects cointegration at 99% probability if score < crit_value[0], 95% if score < crit_value[1], 90% if score < crit_value[2]

In [7]:
score, pvalue, crit_value = coint( 
    prices["NVDA"], 
    prices["AMD"],
    trend="c",
    autolag="AIC"
)

print(f"Score: {score}")
print(f"P-value: {pvalue}")
print(f"Critical Value {crit_value}")

Score: -2.184391539266396
P-value: 0.4324264674310532
Critical Value [-3.896468   -3.33614562 -3.04446084]


In [8]:
if score < crit_value[0]:
    print("There is a 99% chance of cointegration!")
elif score < crit_value[1]:
    print("There is a 95% chance of cointegration!")
elif score < crit_value[2]:
    print("There is a 90% chance of cointegration!")
else:
    print("There is no statistically significant cointegration in the dataset. ")

There is no statistically significant cointegration in the dataset. 
